# 15 FS4 Huang-Style Feature Construction Plan

This notebook is the **feature-construction planning skeleton** for the future `FS4` workstream.

Its job is to translate the frozen retrieval logic from notebook `14` into a clear plan for:
- which similar-day-derived price features should exist
- what the first-pass versus extended `FS4` scope should be
- how the future feature store should fit the current rolling-origin pipeline
- what leakage and validation checks must be satisfied before any model run is allowed

This notebook must stop at planning. It must **not** build the final FS4 feature store or launch training.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
fs4_methodology_doc = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices/docs/fs4_huang_style_methodology_plan.md"

print(fs4_methodology_doc)


## 1. Notebook Intent

Notebooks `13` and `14` are meant to freeze:
- causal similarity-driver families
- local-day profile rules
- candidate-window logic
- the first-pass retrieval family

This notebook should then answer the next question:
- once similar days have been retrieved causally, which **derived price features** should actually be created for the later `LEAR` and `XGBoost` comparison under the shared walk-forward pipeline?


## 2. Future FS4 Feature Construction Categories

The future executed version of this notebook should group FS4 features into clear categories.

### A. Direct top-`K` similar-day price profiles
- same-hour price from the top-1 similar day
- same-hour price from the top-2 similar day
- same-hour price from the top-3 or top-5 similar day if retained

### B. Weighted-average similar-day price profile
- hour-wise weighted average across the top-`K` similar days

### C. Cluster-conditioned weighted-average profile
- weighted average computed only from the dominant historical price cluster among the retrieved candidates

### D. Similarity-score metadata
- top-1 score
- mean top-`K` score
- top-1 minus top-2 score gap
- age in days of the top similar day
- candidate concentration by cluster or recency bucket

### E. Distributional summaries across similar days
- per-hour median
- per-hour lower and upper quantiles
- per-hour spread or interquartile range


## 3. Simple Versus Extended FS4

Recommended first-pass `FS4` should stay small and defensible.

### First-pass operational scope
- feature-based similarity as the main retrieval rule
- `90`-day candidate window
- direct top-1 similar-day price profile
- weighted-average top-`K` similar-day price profile
- a small metadata block with similarity strength and top-candidate age

### Extended scope for later phases only
- aggregate-weighted similarity family
- cluster-conditioned weighted averages
- residual-style similarity drivers
- cross-border similarity families
- quantile and dispersion summaries across similar days
- explicit annual-anchor extensions if validation evidence supports them

The guiding principle is transparency and comparability, not maximal feature count.


## 4. Schema Expectations For A Future FS4 Store

The future FS4 artifacts should be compatible with the existing long-format forecasting pipeline.

Minimum planning fields for a future similar-day candidate log:
- `model` or retrieval-family label
- `fs_level = FS4`
- `forecast_origin_utc`
- `target_local_date`
- `target_hour_local`
- `candidate_local_date`
- `candidate_rank`
- `similarity_family`
- `similarity_score`
- `candidate_age_days`
- `candidate_cluster_id` if applicable
- `profile_quality_flag`
- `imputed_hour_count`

Minimum planning fields for a future derived-feature table:
- `forecast_origin_utc`
- `target_timestamp_utc`
- `lead_day`
- `feature_name`
- `feature_value`
- `feature_family = FS4`
- `similarity_family_source`
- `construction_variant`

TODO for later execution:
- decide whether the implementation stores a wide model matrix, a normalized long feature log, or both
- keep provenance fields so every derived feature can be traced back to the retrieved candidate days


## 5. Compatibility Notes With The Current Walk-Forward Pipeline

The later FS4 implementation must preserve the current benchmark architecture:
- same splits
- same daily rolling-origin refit
- same forecast origin
- same `D..D+4` horizon definition
- same output schema for predictions and timing

Recommended first-pass compatibility design:
- treat similar-day construction primarily as a `D`-only local-day problem first
- if the first operational FS4 execution is launched, it should likely mirror the existing branching logic: richer `D`-only features with a guidance-safe backbone for `D+1..D+4`
- only after that should a full-horizon-safe FS4 extension be attempted with families known beyond day `D`

This keeps the eventual comparison against `FS2` and `FS3` fair and methodologically clear.


## 6. Validation And Leakage Checklist

Before any future FS4 run is allowed, the implementation should pass a written checklist.

Required items:
- every target-day similarity input satisfies `known_at_utc <= forecast_origin_utc`
- every derived feature inherits a valid causal provenance chain
- gap-filled target values are used only for feature construction, never for scoring
- all lookback-window and top-`K` design choices are frozen on train plus validation only
- the final test period remains untouched until the methodology is frozen
- no retrieved candidate day is drawn from the future relative to the origin
- any cluster-conditioned feature uses only historical candidate-day cluster labels, never the unknown future target-day price cluster

The notebook should later keep this checklist visible so the thesis can defend the causal validity of the whole FS4 pipeline.


## 7. Later Reporting Requirements

When `FS4` is eventually executed, results should be reported in two complementary styles.

### Project-standard benchmark style
- same walk-forward comparison structure as the rest of the thesis
- `MAE`, `RMSE`, `bias`, `rMAE`
- Diebold-Mariano test where feasible
- runtime and fit-time reporting per origin

### Huang-like interpretive style
- workflow figure or structured methodology summary
- within-day feature and target distributions
- descriptive-statistics plus correlation table
- cluster figure plus summary table
- similar-day window-analysis figure
- example top similar-day tables
- recency-bucket proportion table
- final methodological decision table

Placeholders only for later phases:
- selected-feature importance table
- model-comparison metrics table
- selected case-day forecast plots


## 8. Explicit Non-Goals For This Notebook

This planning notebook should stop before:
- building the FS4 feature store
- wiring FS4 into LEAR or XGBoost
- running benchmark suites
- writing official comparison outputs

The present task is to make those later steps **methodologically obvious**, not to execute them.


In [ ]:
FS4_NOTEBOOK_15_SCHEMA_PLAN = {
    "candidate_log_required_fields": [
        "forecast_origin_utc",
        "target_local_date",
        "target_hour_local",
        "candidate_local_date",
        "candidate_rank",
        "similarity_family",
        "similarity_score",
        "candidate_age_days",
        "profile_quality_flag",
    ],
    "derived_feature_required_fields": [
        "forecast_origin_utc",
        "target_timestamp_utc",
        "lead_day",
        "feature_name",
        "feature_value",
        "feature_family",
        "construction_variant",
    ],
    "recommended_first_pass_features": [
        "top1_similar_day_price_profile",
        "weighted_average_topk_price_profile",
        "top1_similarity_score",
        "mean_topk_similarity_score",
        "top1_candidate_age_days",
    ],
}

FS4_NOTEBOOK_15_TODOS = [
    "Translate the frozen retrieval rule into concrete derived price features.",
    "Decide the future storage layout for candidate logs and derived features.",
    "Write the leakage checklist into the eventual implementation notebook or module docs.",
    "Prepare later reporting templates without running models yet.",
]

pd.DataFrame(
    {
        "section": list(FS4_NOTEBOOK_15_SCHEMA_PLAN.keys()),
        "value": list(FS4_NOTEBOOK_15_SCHEMA_PLAN.values()),
    }
)
